In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from sys import prefix
from utils import MusicDBPermDir
from pandas import Series, DataFrame
mp = MasterParams(verbose=True)
io = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import lastfm
mio   = lastfm.MusicDBIO(verbose=False)
apiio = lastfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("LastFM")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant LastFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/LastFM


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtistData   = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
localArtistInfo    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistInfo".format(db.lower()))
oldToNew           = MusicDBData(path=permDir, fname="{0}oldToNewMap".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Artist Search Data:        {0}".format(len(searchArtistData)))
#print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist Info:         {0}".format(len(localArtistInfo.get())))
print("   Old => New Map:            {0}".format(len(oldToNew.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

LastFM Search Results
   Global Master Search:      140121
   Global Master Search Data: 0
   Artist Search Data:        2197579
   Local Artist Info:         353454
   Old => New Map:            65023
   Local Album Search:        190415
   Errors:                    0
   Known Summary IDs:         138029


# Move Files Around

In [ ]:
aid = lastfm.MusicDBID()

In [ ]:
aid = lastfm.MusicDBID()
#from tqdm.notebook.*
from tqdm._tqdm_notebook import tqdm_notebook
tqdm_notebook.pandas()
searchArtistData   = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()
searchArtistData["NAME"] = searchArtistData['name'].apply(aid.getArtistIDName)
#searchArtistData["ArtistPseudoIDOld"] = searchArtistData['name'].progress_apply(aid.getArtistPseudoIDOld)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
#searchArtistData["ArtistPseudoID"] = searchArtistData['name'].progress_apply(aid.getArtistPseudoID)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
#possibleArtistsData = searchArtistData['name']
#possibleArtistsData.index = searchArtistData['ArtistPseudoID']

## Artist

In [ ]:
aid = lastfm.MusicDBID()
#localArtistsInfoDict = localArtistInfo.get()
#oldToNewMap = {}
print("Local Artist Info:         {0}".format(len(localArtistsInfoDict)))
print("="*175)
for modVal in range(100):
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}".format(modVal)).glob("*.p")
    for i,ifile in enumerate(files):
        oldID = ifile.stem
        name = io.get(ifile).get('name')
        psid = aid.getArtistPseudoID(name)
        if psid is None:
            continue
        oldToNewMap[oldID] = psid
        artistInfoFilenameOld = FileInfo(ifile)
        artistInfoFilenameNew = mio.data.getRawArtistInfoFilename(mio.mv.get(psid), psid)
        localArtistsInfoDict[psid] = name
        artistInfoFilenameOld.mvFile(artistInfoFilenameNew)
        if i % 100 == 0:
            print("="*150)
            print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
            localArtistInfo.save(data=localArtistsInfoDict)
            print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
            io.save(idata=oldToNewMap, ifile="oldToNewMap.p")            
            print("="*150)

    print("="*150)
    print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
    localArtistInfo.save(data=localArtistsInfoDict)
    print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
    io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)

In [ ]:
artistInfoData = io.get(ifile)

In [ ]:
localArtistsInfoDict = localArtistInfo.get()
missing = io.get('missing.p')
print("Local Artist Info:         {0}".format(len(localArtistsInfoDict)))
print("Missing Artist Map:        {0}".format(len(missing)))
print("="*175)
nMiss = 0
for i,(idx,row) in enumerate(searchArtistData.iterrows()):
    psidOld = aid.getArtistPseudoIDOld(row["name"])
    psid    = aid.getArtistPseudoID(row["name"])
    if localArtistsInfoDict.get(psid) is not None:
        continue
    if missing.get(psid) is not None:
        continue
    name    = row["name"]
    artistInfoFilenameOld = mio.data.getRawArtistInfoFilename(mio.mv.get(psidOld), psidOld)
    artistInfoFilenameOld = FileInfo(artistInfoFilenameOld.str.replace("lastfm", "lastfmold"))
    if artistInfoFilenameOld.exists():
        artistInfoFilenameNew = mio.data.getRawArtistInfoFilename(mio.mv.get(psid), psid)
        #print("{0: <50}{1: >20}{2: >70}{3: >10}".format(name,psidOld,artistInfoFilenameOld.str,artistInfoFilenameOld.exists()))
        #print("{0: <50}{1: >20}{2: >70}{3: >10}".format(row["NAME"],psid,artistInfoFilenameNew.str,""))
        #print("{0: <50}{1: <20}".format(name,psid),end="")
        localArtistsInfoDict[psid] = name
        artistInfoFilenameOld.mvFile(artistInfoFilenameNew)
        nMiss = 0
    else:
        missing[psid] = name
        print("{0: <50}{1: <20} Missing".format(name,psid))
        nMiss += 1
        
    if nMiss > 1000:
        break
        
    if i % 2500 == 0:
        print("="*150)
        print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
        localArtistInfo.save(data=localArtistsInfoDict)
        print("Saving {0} Missing Artist ID Lookups".format(len(missing)))
        io.save(idata=missing, ifile="missing.p")
        print("="*150)
        
        
print("Saving {0} Known Artist ID Lookups".format(len(localArtistsInfoDict)))
localArtistInfo.save(data=localArtistsInfoDict)
print("Saving {0} Missing Artist ID Lookups".format(len(missing)))
io.save(idata=missing, ifile="missing.p")

## Albums

In [ ]:
aid = lastfm.MusicDBID()
localAlbumsDict = localAlbums.get()
localArtistsInfoDict = localArtistInfo.get()
oldToNewMap = oldToNew.get()
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Old => New Map:       {0}".format(len(oldToNewMap)))
print("="*175)
newData = {}
for modVal in range(2,100):
    newData[modVal] = 0
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p")
    print("="*150)

    for i,ifile in enumerate(files):
        oldID = ifile.stem
        psid = oldToNewMap.get(oldID)
        if psid is None:
            continue
        name = localArtistsInfoDict.get(psid)
        if name is None:
            continue
        albumsFilenameOld = FileInfo(ifile)
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        newData[modVal] += 1
        if (i+1) % 100 == 0:
            print("  ===> Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
            localAlbums.save(data=localAlbumsDict)
    print("="*150)
    print("Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
    localAlbums.save(data=localAlbumsDict)
    print("="*150)
    print("")

In [ ]:
aid = lastfm.MusicDBID()
localAlbumsDict = localAlbums.get()
localArtistsInfoDict = localArtistInfo.get()
oldToNewMap = oldToNew.get()
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Old => New Map:       {0}".format(len(oldToNewMap)))
print("="*175)
newData = {}
for modVal in range(100):
    newData[modVal] = 0
    files = DirInfo("/Volumes/Piggy/Discog/artists-lastfmold/{0}/albums".format(modVal)).glob("*.p")
    print("="*150)

    for i,ifile in enumerate(files):
        oldID = ifile.stem
        psid = oldToNewMap.get(oldID)
        name = localArtistsInfoDict.get(psid) if psid is not None else None
        if name is None:
            data = io.get(ifile)
            cntr = Counter()
            for k,v in data.items():
                for rec in v:
                    cntr[rec["artistName"]] += 1
            if len(cntr) > 0:
                name = cntr.most_common()[0][0]
                psid = aid.getArtistPseudoID(name)
                oldToNewMap[oldID] = psid
            else:
                continue
        albumsFilenameOld = FileInfo(ifile)
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        newData[modVal] += 1
        if (i+1) % 100 == 0:
            print("  ===> Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
            localAlbums.save(data=localAlbumsDict)
            print("  ===> Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
            io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)
    print("Saving {0} [{1}/{2}] Known Artist ID Lookups".format(len(localAlbumsDict), newData[modVal], sum(newData.values())))
    localAlbums.save(data=localAlbumsDict)
    print("Saving {0} Old=>New Artist ID Lookups".format(len(oldToNewMap)))
    io.save(idata=oldToNewMap, ifile="oldToNewMap.p")                
    print("="*150)
    print("")

In [ ]:
localAlbumsDict = localAlbums.get()
missingAlbums = io.get('missingAlbums.p')
print("Local Albums:         {0}".format(len(localAlbumsDict)))
print("Missing Album Map:    {0}".format(len(missingAlbums)))
print("="*175)
nMiss = 0

for i,(idx,row) in enumerate(searchArtistData.iterrows()):
    psidOld = aid.getArtistPseudoIDOld(row["name"])
    psid    = aid.getArtistPseudoID(row["name"])
    if localAlbumsDict.get(psid) is not None:
        continue
    if missingAlbums.get(psid) is not None:
        continue
    name    = row["name"]
    albumsFilenameOld = mio.data.getRawArtistAlbumFilename(mio.mv.get(psidOld), psidOld)
    albumsFilenameOld = FileInfo(albumsFilenameOld.str.replace("lastfm", "lastfmold"))
    if albumsFilenameOld.exists():
        albumsFilenameNew = mio.data.getRawArtistAlbumFilename(mio.mv.get(psid), psid)
        localAlbumsDict[psid] = name
        albumsFilenameOld.mvFile(albumsFilenameNew, debug=False)
        nMiss = 0
    else:
        missingAlbums[psid] = name
        print("{0: <50}{1: <20} Missing".format(name,psid))
        nMiss += 1
        
    if nMiss > 1000:
        break
        
    if i % 1000 == 0:
        print("="*150)
        print("Saving {0} Known Artist ID Lookups".format(len(localAlbumsDict)))
        localAlbums.save(data=localAlbumsDict)
        print("Saving {0} Missing Artist ID Lookups".format(len(missingAlbums)))
        io.save(idata=missingAlbums, ifile="missingAlbums.p")
        print("="*150)
        

print("Saving {0} Known Album Artist ID Lookups".format(len(localAlbumsDict)))
localAlbums.save(data=localAlbumsDict)
print("Saving {0} Missing Album Artist ID Lookups".format(len(missingAlbums)))
io.save(idata=missingAlbums, ifile="missingAlbums.p")

# Download Artist Search Data

In [ ]:
mio   = lastfm.MusicDBIO(verbose=False)
apiio = lastfm.RawAPIData(debug=True)

## Find Artists To Download

In [ ]:
from musicdb import MusicDBIO
mdbio = MusicDBIO()
mmeDF = mdbio.getData()
unmatchedArtistNames = Series(mmeDF["ArtistName"].unique())
searchedForMasterArtists = masterArtists.get()
artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(len(unmatchedArtistNames)))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "2:30pm")
tt = TermTime("today", "9:15pm")
n  = 0

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if searchedForMasterArtists.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForMasterArtists[artistName] = True
        apiio.sleep(3.5)
        continue
        
    searchedForMasterArtistsData[artistName] = response
    searchedForMasterArtists[artistName] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = DataFrame(getFlatList([v for k,v in mad.items()]))
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['listeners'] = searchDF['listeners'].astype(int)
    searchDF = searchDF.sort_values(by='listeners', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    lastfm.MusicDBIO(verbose=False).data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
masterArtistsData.save(data={})

# Download Artist Info Data

In [6]:
mio   = lastfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = lastfm.RawAPIData(debug=True)

LastFM API(Key=0017f8ea36758d0923766b8121a98984)


## Find Artists To Download

In [7]:
useSearchArtists = True
useMissingName   = False
useNonOverlap    = False

if useSearchArtists is True:
    aid = lastfm.MusicDBID()
    if "ArtistPseudoID" not in searchArtistData.columns:
        searchArtistData["ArtistPseudoID"] = searchArtistData['name'].apply(aid.getArtistPseudoID)  #searchArtistData["ArtistID"] = searchArtistData['url'].apply(aid.getArtistID)
    possibleArtistsData = searchArtistData['name']
    possibleArtistsData.index = searchArtistData['ArtistPseudoID']
    localArtistInfoDict = localArtistInfo.get()
    artistInfoToGet = possibleArtistsData[~possibleArtistsData.index.isin(localArtistInfoDict.keys())]
    
    print("{0} Search Results".format(db))
    print("   Possible Artist Info:   {0}".format(possibleArtistsData.shape[0]))
    print("   Downloaded Artist Info: {0}".format(len(localArtistInfoDict)))
    print("   Artist Info To Get:     {0}".format(len(artistInfoToGet)))
elif useMissingName is True:
    noNameArtists = knownArtists[knownArtists.str.len() < 1].index
    artistInfoToGet = noNameArtists[~artistIDsToGet.isin(localArtistIDs.get().keys())]
    
    print("{0} Search Results".format(db))
    print("   Known Summary IDs:    {0}".format(len(knownArtists)))
    print("   No Name Artists:      {0}".format(len(noNameArtists)))
    print("   Artists Info To Get:  {0}".format(len(artistInfoToGet)))
elif useNonOverlap is True:
    sAlbums  = Series(localAlbums.get())
    sArtists = Series(localArtistInfo.get())
    artistInfoToGet = sAlbums[~sAlbums.index.isin(sArtists.index)]
    #artistsWithoutAlbums = sArtists[~sArtists.index.isin(sAlbums.index)]
    
    print("{0} Search Results".format(db))
    print("   Known Artist Infos:   {0}".format(len(sArtists)))
    print("   Known Album Artists:  {0}".format(len(sAlbums)))
    print("   Artists Info To Get:  {0}".format(len(artistInfoToGet)))
    
    
#   Artist Info To Get:     1927905
#   Artist Info To Get:     1910283
#   Artist Info To Get:     1885415
#   Artist Info To Get:     1861401
#   Artist Info To Get:     1801866

LastFM Search Results
   Possible Artist Info:   2197579
   Downloaded Artist Info: 353454
   Artist Info To Get:     1801866


## Download Data

In [8]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "5:30pm")
#tt = TermTime("today", "7:15pm")
#tt = TermTime("today", "9:00pm")
n  = 0

searchedForLocalArtistInfo = localArtistInfo.get()
searchedForErrors          = errors.get()
for i,(artistPseudoID,artistName) in enumerate(artistInfoToGet.iteritems()):
    if searchedForLocalArtistInfo.get(artistPseudoID) is not None:
        continue
    if searchedForErrors.get(artistPseudoID) is not None:
        continue

    modVal=mio.mv.get(artistPseudoID)
    if mio.data.getRawArtistInfoFilename(modVal, artistPseudoID).exists():
        searchedForLocalArtistInfo[artistPseudoID] = artistName
        continue

    response = apiio.getArtistInfo(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForErrors[artistPseudoID] = artistName
        apiio.sleep(3.5)
        continue
        
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=artistPseudoID)        
    searchedForLocalArtistInfo[artistPseudoID] = artistName
    apiio.sleep(2)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist Infos".format(len(searchedForLocalArtistInfo), db))
        localArtistInfo.save(data=searchedForLocalArtistInfo)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0}] {1} Searched For Artist IDs".format(len(searchedForLocalArtistInfo), db))
localArtistInfo.save(data=searchedForLocalArtistInfo)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting LastFM ArtistIDs]    ==> Time Is 2022-03-19 21:29:26
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-03-20 11:00:00 <====
   ====> Will Terminate Process 13 Hours and 30 Minutes From Now
Searching For The Happy Talk Band                               1  [9]
Searching For Dennis DJ & MC Don Juan                           1  [9]
Searching For Jehann Corvus                                     1  [9]
Searching For Photographer vs. Josh Wink                        1  [9]
Searching For The Crystal Method feat. Stefanie King Warfield   1  [9]
Searching For The Utopiates                                     1  [9]
Searching For DJ Phantasy & Dextone                             1  [9]
Searching For Tarrot                                            1  [9]
Searching For Blade Runner OST                                  1  [9]
Searching For C + C Music Factory presents Freedom Williams     1  [9]
S

Searching For Noriko Awaya                                      1  [9]
Searching For Dru Hill, Will Smith & Kool Mo Dee                1  [9]
Searching For Sapardi Djoko Damono                              1  [9]
Searching For King General Bucks Up Pon De Bush Chemists        1  [9]
Searching For Major Lazer & Mungo's Hi Fi                       1  [9]
Searching For How Larry Heard Made House Music Deep             1  [9]
Searching For Anouk - Hotel New York                            1  [9]
Searching For Chris Watson & Marcus Davidson                    1  [9]
Searching For Black Coffee, Diplo, Elderbrook                   1  [9]
Searching For Jamie Torres Y Alejandro Seoane                   1  [9]
100/?      : Process [Getting LastFM ArtistIDs] Has Run For 4 Minutes and 20 Seconds.
Saving 353554 LastFM Searched For Artist Infos
Searching For Marcos Valle e Marcelo Camelo                     1  [9]
Searching For Malente & ManroX                                  1  [9]
Searching For A

Searching For Ten Years A Day                                   1  [9]
Searching For Augustana Choir                                   1  [9]
Searching For GOT7 “Stop stop it(하지하지마)” M                      1  [9]
Searching For John Joseph Woods                                 1  [9]
Searching For King Krule @ Primavera Sound                      1  [9]
Searching For Plague throat                                     1  [9]
Searching For Ghostface Killah, GZA, Cappadonna, Masta Killa, Raekwon1  [9]
Searching For Kavinsky Feat. Havoc                              1  [9]
Searching For You Make My Dreams Come True                      1  [9]
Searching For Shagy vs. Vanessa Paradise                        1  [9]
Searching For Against The Current & The Ready Set               1  [9]
Searching For Kode 9 Feat the Spaceape                          1  [9]
Searching For Danny Kaye, Louis Armstrong                       1  [9]
Searching For Elissa Kroll                                      1  [9]
S

Searching For Aki Sirkesalo And Lisa Nilsson                    1  [9]
Searching For Bassmonkeys Ft. Abigail Bailey                    1  [9]
Searching For Le Truk, Noggano                                  1  [9]
Searching For www.GrWarez.com                                   1  [9]
Searching For Fray (The)                                        1  [9]
Searching For Colin Firth, Pierce Brosnan, Stellan Skarsgard, Amanda Seyfried & Meryl Stre1  [9]
Searching For Matthew Shipp String Trio                         1  [9]
Searching For Ludwig Güttler, Trumpet; Friedrich Kircheis, Organ1  [9]
Searching For Marc E. Bassy feat. G-Eazy                        1  [9]
Searching For Jack Eric Williams, Angela Lansbury, Ken Jennings 1  [9]
Searching For Rohff - Karim Benzema                             1  [9]
Searching For James and the Drifters                            1  [9]
Searching For Lord Gorgoroth                                    1  [9]
Searching For 6 Prong Paw                          

Searching For Eleven & Chris Cornell                            1  [9]
Searching For Anjo Feat Suzie Kerstegens                        1  [9]
Searching For The Pete Green Corporate Juggernaut               1  [9]
Searching For Shyne, Barrington Levy                            1  [9]
Searching For Will Atkinson Presents Darkboy                    1  [9]
Searching For Coldplay vs. Two Door Cinema Club                 1  [9]
375/?      : Process [Getting LastFM ArtistIDs] Has Run For 16 Minutes and 29 Seconds.
Saving 353829 LastFM Searched For Artist Infos
Searching For HEIST feat MC FATS                                1  [9]
Searching For Willbe (William Lamy)                             1  [9]
Searching For Chorus - Pocahontas, David Ogden Stiers, Jim Cummings & Judy Kuhn1  [9]
Searching For Skepta, D Double E, ASAP Nast                     1  [9]
Searching For Para Wino & Anja Orthodox [Punk Ofiary]           1  [9]
Searching For Partyraiser & Darkcontroller                      1  [9]

...

Searching For Jack Jango                                        1  [9]
Searching For Madonna & Modern Talking                          1  [9]
Searching For Shakira, Carlinhos Brown                          1  [9]
Searching For Crazy Baby Produções                              1  [9]
Searching For ¿Fung Wah Bus?                                    1  [9]
Searching For DHT ft. Emdée                                     1  [9]
Searching For Jenny Hélia & Alibert                             1  [9]
Searching For Wilco & Bob Weir                                  1  [9]
Searching For Volbeat, Neil Fallon                              1  [9]
Searching For Jazzy Bazz/L'Etrange/Nekfeu/Alpha Wann            1  [9]
Searching For Kansas & Will Ferrell                             1  [9]
Searching For Of Mice and Mental Arithmetic                     1  [9]
Searching For Andris Nelsons / Boston Symphony Orchestra        1  [9]
Searching For OG Bobby Johnson                                  1  [9]
Search

Searching For Qadafee & Djiva                                   1  [9]
Searching For Bob Sinclar & Cutee B featuring Gary Pine & Dollarman1  [9]
Searching For Matt Shadetek & Lamin Fofana                      1  [9]
18350/?    : Process [Getting LastFM ArtistIDs] Has Run For 13 Hours and 19 Minutes.
Saving 371804 LastFM Searched For Artist Infos
Searching For Shindy Feat. Kollegah                             1  [9]
Searching For Crossover Kidz                                    1  [9]
Searching For External Subway                                   1  [9]
Searching For Jim Hall & Bill Evans                             1  [9]
Searching For Billie Holiday with Teddy Wilson & His Orchestra/Billie Holiday/Teddy Wilson & His Orchestra1  [9]
Searching For Mitch Miller and The Sing-Along Gang              1  [9]
Searching For The Royal Philharmonic Orchestra  Louis Clark conducting1  [9]
Searching For Perfume Genius & Planningtorock                   1  [9]
Searching For Wielka Kupa Gówna     

Searching For Moira Craig                                       1  [9]
Searching For Sam F ft. The Lonely Island & Lil Jon             1  [9]
Searching For Kanye West Ft. John Legend, The-Dream, Ryan Leslie, Tony Williams, Charlie Wilson, Elly Jackson, Alicia Keys, Fergie, Kid Cudi, Rihanna & Elton John1  [9]
Searching For Desiinger                                         1  [9]
Searching For Emicida, Zeca Pagodinho, Tokyo Ska Paradise Orchestra1  [9]
Searching For Fatos Shabani                                     1  [9]
Searching For Charlie Johnson And His Orchestra                 1  [9]
Searching For Jack Johnson, Donavon Frankenreiter e G. Love     1  [9]
Searching For David Darling/Wulu Bunun                          1  [9]
Searching For Lost Soul (CH)                                    1  [9]
Searching For Carl Cox & Eric Powell                            1  [9]
Searching For SQUEEZE TARELA                                    1  [9]
Searching For Keith Richards / Toots & the Mayt

Searching For deeeznvtzzz                                       1  [9]
Searching For Dr.Alban & Victor                                 1  [9]
Searching For Bizarrap x Duki x Nicki Nicole                    1  [9]
Searching For Giant Gutter From Outer Space                     1  [9]
Searching For DJ Stormtrooper vs DJ D-Lyte feat. MC Night       1  [9]
Searching For Radiohead — The Present Tense                     1  [9]
Searching For Claptone feat. George Kranz                       1  [9]
Searching For Julieta Venegas & Mala Rodríguez                  1  [9]
Searching For Michael Burkat & Lars Klein                       1  [9]
Searching For Prince William                                    1  [9]
Searching For GallantHorn                                       1  [9]
Searching For David Mingyue Liang                               1  [9]
Searching For The Vostok All Stars                              1  [9]
Searching For Tim Deluxe Feat. Terra Deva                       1  [9]
Search

Searching For Taigan Sunset                                     1  [9]
Searching For CIVIL DEFENCE PROGRAMME                           1  [9]
18625/?    : Process [Getting LastFM ArtistIDs] Has Run For 13 Hours and 30 Minutes.
Saving 372079 LastFM Searched For Artist Infos
   ====> Terminate Time [2022-03-20 11:00:00] Is Reached <====
Process [Getting LastFM ArtistIDs] Ran For 13 Hours and 30 Minutes    ==> Time Is 2022-03-20 11:00:19
Saving [372079] LastFM Searched For Artist IDs


In [9]:
""" MusicDBIO Utilities """

__all__ = ["moveLocalFiles"]

from fileutils import FileInfo
from master import MasterParams
from lib.lastfm import MusicDBIO

def moveLocalFiles():
    mp        = MasterParams()
    mioLocal  = MusicDBIO(local=True,mkDirs=True,debug=True)
    mioGlobal = MusicDBIO(local=False,mkDirs=True,debug=True)
    for modVal in mp.getModVals():
        files = mioLocal.dir.getRawAlbumModValDataDir(modVal).glob("*.p")
        _ = [FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistAlbumFilename(modVal,ifile.stem))) for ifile in files]

In [12]:
modVal = 0
mioLocal  = MusicDBIO(local=True,mkDirs=True,debug=True)
files = mioLocal.dir.getRawAlbumModValDataDir(modVal).glob("*.p")
len(list(files))
files = mioLocal.dir.getRawAlbumModValDataDir(modVal).glob("*.p")
len(list(files))

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb
MusicDBBaseDirs(db=LastFM)
   Using Local Path For Raw Data <<====
   RawDataDir     = /Users/tgadfort/Music/Discog/artists-lastfm
   ModValDataDir  = /Volumes/Seagate/Discog/artists-lastfm-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-lastfm-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-lastfm
LastFM SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> Link
ParseRawDataUtils(mdbdata, mdbdir) [LastFM]
LastFM ModValMetaData
  ==> Basic (Universal)
  ==> Media (Media)
  ==> Genre
  ==> Link
  ==> Metric
Running glob(/Users/tgadfort/Music/Discog/artists-lastfm/0/albums/*.p)


0

# Download Album Data

## Find Albums To Get

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(searchedForAlbums.keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "4:00pm")
n  = 0
searchedForAlbums = getSearchedForLocalAlbums()
searchedForErrors = getSearchedForErrors()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        saveSearchedForLocalAlbums(searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

# Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))

# Download Artist Info Data

## Determine Artists To Download

## Download Artist Info

# Download Artist Albums Data

## Determine Artists To Download

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

In [ ]:
searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = spotifyArtists[~spotifyArtists.index.isin(searchedForArtists.index)]['name']
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
from pandas import Series
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    sids = [sid for sid in sids if len(sid) == 22]
    spotifyToGet.update({sid: name for sid in sids})
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Spotify")
api  = apig.getDBAPIData("Spotify")

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
ts = timestat("Downloading Spotify Data")
#tt = termTime("today", "10:00pm")
tt = termTime("tomorrow", "10:00pm")
#tt = termTime("tomorrow", "7:00am")
#tt = termTime("today", "9:30am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistAlbumsKnown(dbID) or True:
        response = api.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistAlbums(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        print("="*150)
        api.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        api.sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

# Extra

In [ ]:
  
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)

In [ ]:
######################################################
# Juypter
######################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from ioUtils import getFile, saveFile
from webUtils import getHTML
from searchUtils import findExt
from fsUtils import setFile, isFile, fileUtil
from fileUtils import getBaseFilename
from timeUtils import timestat
from fileIO import fileIO
from dbUtils import utilsSpotifyCharts


###########################################################################
## General
###########################################################################
import urllib
from glob import glob
from time import sleep
from io import StringIO
from pandas import Series,DataFrame,concat,read_csv,date_range
from datetime import datetime
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath



######################################################
# Versions
######################################################
import sys
print("Python: {0}".format(sys.version))
import datetime as dt
start = dt.datetime.now()

print("Notebook Last Run Initiated: "+str(start))

# Global

In [ ]:
io = fileIO()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()

from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

# Spotify API

In [ ]:
# !pip install spotipy

In [ ]:
from artistSpotify import artistSpotify
#from artistIDBase import artistIDSpotify
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection


##################################################################################################################
# Base Class
##################################################################################################################
class dbSpotify:
    def __init__(self):
        self.db     = "Spotify"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistSpotify(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        self.manc = masterArtistNameCorrection()
        
        
    def getSearchDir(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return searchDir
        
    def getArtistSearchSavename(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return fileUtil(searchDir).join("{0}.p".format(psID))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
def getArtistTrackResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
            sleep(1.0)
    print(" {0}".format(len(searchResults)))
    return searchResults

In [ ]:
def getArtistAlbumResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    href           = requestResult.get('href')
    artistID       = PurePosixPath(unquote(urlparse(href).path)).parts[3]
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        sleep(3.5)
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
    print(" {0}".format(len(searchResults)))
    
    albums = [spotifyAlbumRecord(album).get() for album in searchResults]
    retval = {'href': href, 'artistID': artistID, 'albums': albums}
    return retval

In [ ]:
def getArtistSearchResults(artistName, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    sleep(0.25)
    result = sp.search(artistName, limit=limit, type='artist')
    
    artists = result.get('artists', {}) if isinstance(result,dict) else {}
    href    = artists.get('href')
    total   = artists.get('total')
    nextURL = artists.get('next')
    prevURL = artists.get('previous')
    items   = artists.get('items', [])
    retval  = [spotifyArtistRecord(item).get() for item in items]
    print(len(retval))
    
    return retval

# Artist Search

In [ ]:
from glob import glob
from masterUtils import masterUtils
mu = masterUtils()
disc = mu.getDisc("Spotify")
ts = timestat("Finding Search Files")
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
ts.stop()

In [ ]:
from fileIO import fileIO
io = fileIO()
#io.save(idata=files, ifile="spotifyFiles.p")
files = io.get("spotifyFiles.p")
len(files)

In [ ]:
from fsUtils import fileUtil, mkDir, dirUtil, moveFile
dbIO = dbSpotify()
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    artistName = fileUtil(ifile).basename
    searchDir = dbIO.getSearchDir(artistName)
    if not searchDir.exists():
        print("Making {0}".format(searchDir))
        mkDir(searchDir)
    savename = dbIO.getArtistSearchSavename(artistName=artistName)
    moveFile(ifile,savename)
    
    if (i+1) % 10000 == 0 or (i+1) == 1000 or (i+1) == 100:
        ts.update(n=i+1, N=len(files))
ts.stop()

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
ts = timestat("Getting Searched For Artists")
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
ts = timestat("Getting Artists To Download")
mmeDF = meu.getDF()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtists)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 2500:
        break
    savefile = fileUtil("spotify/artists").join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if savefile.exists():
        continue
    if nErr >= 5:
        break
    result = getArtistSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(2.5)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
    
    if (i+1) % 100 == 0:
        sleep(120)
ts.stop()

## Artist Results

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()

# Move Artist Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile

files = glob("spotify/artists/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    dst = dirUtil("/Volumes/Piggy/Discog/artists-spotify/artists").join(src.name)
    if dst.exists():
        continue
    moveFile(src.path, dst)

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

# Move Album Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile,mkDir
from artistModValue import artistModValue
amv = artistModValue()

files = glob("spotify/albums/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    albumsData = io.get(ifile)
    artistID = albumsData['artistID']
    modValue = amv.getModVal(artistID)
    modDir = dirUtil("/Volumes/Piggy/Discog/artists-spotify/").join(str(modValue))
    if not modDir.exists():
        mkDir(modDir)
    dst = dirUtil(modDir).join("{0}.p".format(artistID))
    if dst.exists():
        continue
    print("  ==>  {0}".format(dst))
    moveFile(src.path, dst)

# Create Master Artist ID List

In [ ]:
from glob import glob
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
artistsData = []
N = len(files)
ts = timestat("Loading {0} Spotify Artist Search Files".format(N))
for i,ifile in enumerate(files):    
    artistsData += io.get(ifile)
    if (i+1) % 5000 == 0 or (i+1) == 1000:
        ts.update(n=i+1, N=N)
ts.stop()

In [ ]:
def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

df = DataFrame(artistsData)
df["followers"] = df["followers"].apply(getFollowers)
df.index = df['sid']
df.drop(['sid'], axis=1, inplace=True)
df = df[~df.index.duplicated(keep='first')]
df = df.sort_values(by='popularity', ascending=False)

print("Saving {0} Spotify Artists Data".format(df.shape[0]))
io.save(idata=df, ifile="/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
df.head(10)

In [ ]:
print(df.shape)

# Artist Albums

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

## Download Albums Data

In [ ]:
spotifyArtists['popularity'].hist(bins=100, log='y')

In [ ]:
spotifyArtists[spotifyArtists["popularity"] >= 60].shape

In [ ]:
knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
spotifyToGet = Series(spotifyToGet)
#spotifyToGet = knownSpotify["ArtistName"]
#spotifyToGet.index = list(knownSpotify["Spotify"].values)
#spotifyToGet.name = "name"
#spotifyToGet

In [ ]:
toget = {}
from artistModValue import artistModValue
from masterUtils import masterUtils
from fsUtils import dirUtil, mkDir
mu   = masterUtils()
amv  = artistModValue()
disc = mu.discs["Spotify"]


#spotifyToGet = spotifyArtists[spotifyArtists["popularity"] >= 60]
#print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
#for i,(artistID,artistName) in enumerate(spotifyToGet["name"].iteritems()):

print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
nKnown = 0
for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
    modVal   = amv.getModVal(artistID)
    modDir   = dirUtil(disc.getArtistsDir()).join(str(modVal))
    if not modDir.exists():
        print("Making {0}".format(modDir))
        mkDir(modDir)
    savefile = dirUtil(modDir).join("{0}.p".format(artistID))
    if savefile.exists():
        nKnown += 1
        continue
    #print("{0: <25}{1: <20} ({2})".format(artistName,artistID,modVal))
    toget[savefile] = (artistName,artistID)
    if len(toget) >= 2500:
        break
print("Found {0} Known Artists".format(nKnown))
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(7.5)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(15.0)
        print("")
    
    if (i+1) % 25 == 0:
        sleep(30.0)
ts.stop()

## Determine Artists To Download

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
io.save(idata={}, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
stop = 150
dbIO = dbSpotify()
api  = spotifyAPIIO()
ts   = timestat("Getting Artist Data From Spotify API")
n    = 0

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    savename = dbIO.getArtistAlbumsSavename(artistID)
    if savename.exists():
        searchedForArtists[artistID] = artistName 
        continue
        
    #print(artistID,artistName,savename)
    response = api.getArtistAlbums(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistAlbums(artistID=artistID, artistAlbums=response)
    searchedForArtists[artistID] = artistName    
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Error Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
if False:
    searchedForArtists = io.get("spotifySearchedForArtists.p")
    print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

    ts = timestat("Finding Spotify Saves")
    first = True
    for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
        if searchedForArtists.get(artistID) is not None:
            continue
        if first:
            print("Start: {0}".format(i))
            first = False
        if dbIO.getArtistAlbumsSavename(artistID).exists():
            searchedForArtists[artistID] = artistName
            if len(searchedForArtists) % 5000 == 0:
                break
        else:
            break

        if (i+1) % 100 == 0:
            ts.update(n=i+1,N=len(spotifyToGet))        
            #io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

    ts.stop()
    print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
    io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

In [ ]:
spotifyArtists.head()

In [ ]:
from dbArtistsBase import dbArtistsBase
from fileUtils import getBaseFilename
from fsUtils import isFile, setFile, setDir
from ioUtils import getFile, saveFile
from timeUtils import timestat
from sys import prefix
import urllib
from time import sleep
from webUtils import getHTML
from pandas import Series
    
from fileIO import fileIO
from fsUtils import fileUtil

In [ ]:
from dbArtistsParse import dbArtistsSpotifyAPI
from dbArtistsSpotify import dbArtistsSpotify
dbAP = dbArtistsSpotifyAPI(dbArtistsSpotify())

In [ ]:
for modVal in range(100):
    dbAP.parse(modVal=modVal, expr='< 0 Days', force=False)
    
#for modVal in range(100):
#    dbAP.parseSearch(modVal, expr='< 0 Days', force=False)

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()
modVal = 91
idx = artistSearchData.reset_index()['sid'].apply(amv.getModVal) == modVal
idx.index = artistSearchData.index
artistSearchData[idx]

In [ ]:
artistSearchData.head()

In [ ]:
art = artistSpotify()
art.getData(inputData).show()

In [ ]:
io.get("/Users/tgadfort/Music/Discog/artists-spotify-db/91-DB.p").get('1uNFoZAHBGtllmzznpCI3s').show()

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
mmeDF = meu.getDF()

In [ ]:

toget = {}
for i,(artistID,artistName) in enumerate(spotifyArtists["name"].iteritems()):
    if i >= 5:
        break
    savefile = fileUtil("spotify/albums").join("{0}--{1}.p".format(artistID,manc.clean(artistName)))
    if savefile.exists():
        pass
        #continue
    toget[savefile] = (artistName,artistID)
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(3.0)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

## Artist Albums Results

In [ ]:
from glob import glob
files = glob("spotify/albums/*.p")
artistAlbumsData = {}
for ifile in files:
    albumsData   = io.get(ifile)
    artistID     = albumsData['artistID']
    artistAlbums = albumsData['albums']
    if artistAlbumsData.get(artistID) is not None:
        raise ValueError("Already have data for {0}".format(artistID))
    artistAlbumsData[artistID] = artistAlbums

In [ ]:
retval = getArtistAlbumResults(artistID='46sPgFJpEcoxUJNr1ouR9G', artistName='dummy')

In [ ]:
s = Series(artistAlbumsData)

In [ ]:
s.apply(len)